In [0]:
MERGE INTO pondolens.dev_silver.shared_region_crosswalk AS t
USING (
  WITH cleaned AS (
    SELECT
      TRIM(region) AS source_region,
      TRY_CAST(TRY_CAST(year AS DOUBLE) AS INT) AS year_int
    FROM pondolens.dev_bronze.unemployment_rate
    WHERE LOWER(TRIM(geographic_level)) = 'region'
  ),
  regional_years AS (
    SELECT
      source_region,
      MIN(year_int) AS valid_from_year,
      MAX(year_int) AS valid_to_year
    FROM cleaned
    WHERE year_int BETWEEN 2021 AND 2025
    GROUP BY source_region
  ),
  region_map AS (
    SELECT *
    FROM VALUES
      ('BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO (BARMM)',
       'BARMM', 'BARMM',
       'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO'),

      ('BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO (BARMM/ARMM)',
       'BARMM/ARMM', 'BARMM',
       'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO'),

      ('CORDILLERA ADMINISTRATIVE REGION (CAR)',
       'CAR', 'CAR', 'CORDILLERA ADMINISTRATIVE REGION'),

      ('NATIONAL CAPITAL REGION (NCR)',
       'NCR', 'NCR', 'NATIONAL CAPITAL REGION'),

      ('NEGROS ISLAND REGION (NIR)',
       'NIR', 'NIR', 'NEGROS ISLAND REGION'),

      ('REGION I (ILOCOS REGION)',
       'I', 'I', 'ILOCOS REGION'),

      ('REGION II (CAGAYAN VALLEY)',
       'II', 'II', 'CAGAYAN VALLEY'),

      ('REGION III (CENTRAL LUZON)',
       'III', 'III', 'CENTRAL LUZON'),

      ('REGION IV-A (CALABARZON)',
       'IV-A', 'IVA', 'CALABARZON'),

      ('REGION IVA (CALABARZON)',
       'IVA', 'IVA', 'CALABARZON'),

      ('REGION IVA CALABARZON',
       'IVA', 'IVA', 'CALABARZON'),

      ('MIMAROPA REGION',
       CAST(NULL AS STRING), 'IVB', 'MIMAROPA REGION'),

      ('REGION V (BICOL REGION)',
       'V', 'V', 'BICOL REGION'),

      ('REGION VI (WESTERN VISAYAS)',
       'VI', 'VI', 'WESTERN VISAYAS'),

      ('REGION VII (CENTRAL VISAYAS)',
       'VII', 'VII', 'CENTRAL VISAYAS'),

      ('REGION VIII (EASTERN VISAYAS)',
       'VIII', 'VIII', 'EASTERN VISAYAS'),

      ('REGION IX (ZAMBOANGA PENINSULA)',
       'IX', 'IX', 'ZAMBOANGA PENINSULA'),

      ('REGION X (NORTHERN MINDANAO)',
       'X', 'X', 'NORTHERN MINDANAO'),

      ('REGION XI (DAVAO REGION)',
       'XI', 'XI', 'DAVAO REGION'),

      ('REGION XII (SOCCSKSARGEN)',
       'XII', 'XII', 'SOCCSKSARGEN'),

      ('REGION XIII (CARAGA)',
       'XIII', 'XIII', 'CARAGA')

    AS m(source_region, source_code, region_code, region_name)
  )
  SELECT
    'Philippines' AS country,
    y.source_region,
    m.region_code,
    m.region_name,
    y.valid_from_year,
    y.valid_to_year,
    m.source_code,
    'region' AS geographic_level
  FROM regional_years y
  INNER JOIN region_map m
    ON y.source_region = m.source_region
) AS s

ON LOWER(TRIM(t.country)) = LOWER(s.country)
AND TRIM(t.source_region) = s.source_region
AND (
  LOWER(TRIM(t.geographic_level)) = 'region'
  OR t.geographic_level IS NULL
)

WHEN MATCHED THEN UPDATE SET
  t.region_code = s.region_code,
  t.region_name = s.region_name,
  t.valid_from_year = LEAST(t.valid_from_year, s.valid_from_year),
  t.valid_to_year = GREATEST(t.valid_to_year, s.valid_to_year),
  t.source_code = s.source_code,
  t.geographic_level = s.geographic_level

WHEN NOT MATCHED THEN INSERT (
  country,
  source_region,
  region_code,
  region_name,
  valid_from_year,
  valid_to_year,
  source_code,
  geographic_level
)
VALUES (
  s.country,
  s.source_region,
  s.region_code,
  s.region_name,
  s.valid_from_year,
  s.valid_to_year,
  s.source_code,
  s.geographic_level
);




SELECT
  source_region,
  source_code,
  region_code,
  region_name,
  valid_from_year,
  valid_to_year
FROM pondolens.dev_silver.shared_region_crosswalk
WHERE LOWER(TRIM(geographic_level)) = 'region'
ORDER BY region_code, source_region;



MERGE INTO pondolens.dev_silver.shared_region_crosswalk AS t
USING (
  SELECT
    'Philippines' AS country,
    'CENTRAL OFFICE' AS source_region,
    'CO' AS region_code,
    'CENTRAL OFFICE' AS region_name,
    2021 AS valid_from_year,
    2025 AS valid_to_year,
    CAST(NULL AS STRING) AS source_code,
    'other' AS geographic_level
) AS s
ON LOWER(TRIM(t.country)) = LOWER(s.country)
AND UPPER(TRIM(t.source_region)) = s.source_region

WHEN MATCHED THEN UPDATE SET
  t.region_code = s.region_code,
  t.region_name = s.region_name,
  t.geographic_level = s.geographic_level,
  t.valid_from_year = LEAST(t.valid_from_year, s.valid_from_year),
  t.valid_to_year = GREATEST(t.valid_to_year, s.valid_to_year)

WHEN NOT MATCHED THEN INSERT (
  country, source_region, region_code, region_name,
  valid_from_year, valid_to_year, source_code, geographic_level
)
VALUES (
  s.country, s.source_region, s.region_code, s.region_name,
  s.valid_from_year, s.valid_to_year, s.source_code, s.geographic_level
);


SELECT *
FROM pondolens.dev_silver.shared_region_crosswalk
WHERE region_code = 'CO';




SELECT
  source_region,
  region_code,
  region_name,
  geographic_level
FROM pondolens.dev_silver.shared_region_crosswalk
WHERE geographic_level IS NULL
   OR region_name IS NULL;


   UPDATE pondolens.dev_silver.shared_region_crosswalk
SET geographic_level = CASE
  WHEN region_code IN ('PHL', 'NW') THEN 'nationwide'
  WHEN region_code = 'CO' THEN 'other'
  WHEN region_code IN (
    'BARMM', 'CAR', 'NCR', 'NIR',
    'I', 'II', 'III', 'IVA', 'IVB',
    'V', 'VI', 'VII', 'VIII', 'IX',
    'X', 'XI', 'XII', 'XIII'
  ) THEN 'region'
  ELSE geographic_level
END
WHERE country = 'Philippines'
  AND geographic_level IS NULL;


  SELECT source_region, region_code, region_name, geographic_level
FROM pondolens.dev_silver.shared_region_crosswalk
WHERE geographic_level IS NULL
   OR region_name IS NULL;
